In [4]:
import pandas as pd
import re

try:
    df = pd.read_csv('dataset_human.csv' , sep=',')
    
    # 2. Extrair apenas os textos escritos por humanos - is_ai_generated == 0 (ou ai_generated == False)
    df_humanos = df[df['is_ai_generated'] == 0].copy()
    
    total_humanos_bruto = len(df_humanos)
    print(f"Total de textos estritamente humanos encontrados: {total_humanos_bruto}")
    
    print("A limpar quebras de linha e pontos e vírgulas dos textos")
    
    # 3. Função para limpar cada texto
    def limpar_texto(texto):
        texto = str(texto)
        # Substituir ponto e vírgula por vírgula (para não partir as colunas)
        texto = texto.replace(';', ',')
        # Remover quebras de linha e retornos
        texto = texto.replace('\n', ' ').replace('\r', '')
        # Remover espaços duplos que possam ter ficado
        texto = re.sub(r'\s+', ' ', texto).strip()
        return texto

    df_humanos['abstract_clean'] = df_humanos['abstract'].apply(limpar_texto)
    
    # 4. Contar as palavras e aplicar a régua (80 a 120)
    df_humanos['word_count'] = df_humanos['abstract_clean'].apply(lambda x: len(x.split()))
    
    df_validos = df_humanos[(df_humanos['word_count'] >= 80) & (df_humanos['word_count'] <= 120)].copy()
    
    total_validos = len(df_validos)
    print(f"Total de textos humanos com o tamanho correto (80-120 palavras): {total_validos}")
    
    # Calcular quantos foram deitados fora
    print(f"Textos rejeitados por tamanho incorreto: {total_humanos_bruto - total_validos}\n")
    
    if total_validos > 0:
        # 5. Preparar o formato final para juntar aos outros
        df_final = pd.DataFrame()
        
        # Criar os IDs sequenciais (ex: HUM-0001, HUM-0002...)
        df_final['ID'] = [f"HUM-{i+1:04d}" for i in range(total_validos)]
        
        # Adicionar o texto limpo e a label
        df_final['Text'] = df_validos['abstract_clean'].values
        df_final['Label'] = 'Human'
        
        # 6. Guardar com o separador ';'
        df_final.to_csv('dataset_human_limpo.csv', sep=';', index=False)
        print(f"O ficheiro '{'dataset_human_limpo.csv'}' foi guardado!")
        print(f"Estrutura gerada: {list(df_final.columns)}")
    else:
        print("Nenhum texto cumpriu os requisitos de tamanho.")

except FileNotFoundError:
    print(f"O ficheiro '{'dataset_human.csv'}' não foi encontrado.")
except Exception as e:
    print(f"Ocorreu um erro inesperado: {e}")

Total de textos estritamente humanos encontrados: 2100
A limpar quebras de linha e pontos e vírgulas dos textos
Total de textos humanos com o tamanho correto (80-120 palavras): 892
Textos rejeitados por tamanho incorreto: 1208

O ficheiro 'dataset_human_limpo.csv' foi guardado!
Estrutura gerada: ['ID', 'Text', 'Label']


In [ ]:
ficheiros_csv = [
    'dataset_human_limpo.csv', 
    'dataset_ai.csv'      
]

dfs = []

# Carregar todos os ficheiros da lista
for ficheiro in ficheiros_csv:
    try:
        df_temp = pd.read_csv(ficheiro, sep=';')
        dfs.append(df_temp)
        print(f"Ficheiro '{ficheiro}' carregado com sucesso.")
    except Exception as e:
        print(f"Erro ao ler '{ficheiro}': {e}")

# Juntar tudo num único DataFrame bruto
df_bruto = pd.concat(dfs, ignore_index=True)
print(f"\nTotal de linhas juntas antes da filtragem: {len(df_bruto)}")

# 2. FILTRAGEM 80 a 120 palavras
# Cria uma coluna temporária com a contagem de palavras
df_bruto['WordCount'] = df_bruto['Text'].astype(str).apply(lambda x: len(x.split()))

# Fica apenas com as linhas que estão no range permitido
df_filtrado = df_bruto[(df_bruto['WordCount'] >= 80) & (df_bruto['WordCount'] <= 120)].copy()
linhas_removidas = len(df_bruto) - len(df_filtrado)

print(f"Textos eliminados por não terem entre 80 e 120 palavras: {linhas_removidas}")
print(f"Textos perfeitos que passaram no filtro: {len(df_filtrado)}")

# 3. MISTURAR TUDO (Shuffle)
# frac=1 significa baralhar 100% das linhas; random_state=42 garante reprodutibilidade
df_misturado = df_filtrado.sample(frac=1, random_state=42).reset_index(drop=True)

# 4. REESCREVER OS IDs no formato D1-0001, D1-0002, etc.
df_misturado['ID'] = [f"D1-{i+1:04d}" for i in range(len(df_misturado))]

# 5. LIMPAR E GUARDAR
df_final = df_misturado[['ID', 'Text', 'Label']]

nome_ficheiro_final = 'dataset_final.csv'
df_final.to_csv(nome_ficheiro_final, sep=';', index=False)

print(f"\nO dataset final está guardado como '{nome_ficheiro_final}'.")
print("\nDistribuição final das classes no dataset:")
print(df_final['Label'].value_counts())

Ficheiro 'dataset_human_limpo.csv' carregado com sucesso.
Ficheiro 'dataset.csv' carregado com sucesso.

Total de linhas juntas antes da filtragem: 7492
Textos eliminados por não terem entre 80 e 120 palavras: 494
Textos perfeitos que passaram no filtro: 6998

O dataset final está guardado como 'dataset_final.csv'.

Distribuição final das classes no dataset:
Label
OpenAI       1583
Google       1582
Meta         1544
Anthropic    1397
Human         892
Name: count, dtype: int64
